# ADBE — Economic Value Lens (Capstone)

This is the capstone of Feature 2. The three prior lenses each answered part of the owner's question:

* **2A — Annual snapshot:** did this year's per-share economics improve?
* **2B — Compounding view:** how fast did per-share economics compound over multiple years?
* **2C — Capital allocation:** did management deploy the cash in a way that helped owners?

This notebook synthesizes all three into one deterministic company-level conclusion answering: **if I owned this business privately, what should I know about how its economic value per ownership unit has been developing?**

The synthesis is a documented *hierarchy*, not a score. Long-term compounding anchors the verdict, the latest year confirms or dampens it, capital allocation modifies it, and only severe signals (deepening net debt, sustained ROIC collapse) act as guardrails. Everything else is surfaced as a **watch** signal — economically relevant but not thesis-breaking. There is no 0–100 score and no black-box weighting.

> Set an SEC User-Agent identifying your application and an administrative contact. Replace the sample contact below before making live requests.

In [1]:
import os

from owner_lens import (
    SecClient,
    capital_allocation_from_facts,
    compounding_views_from_facts,
    economic_value_from_facts,
    economic_value_summary_from_facts,
    format_economic_value_summary,
)

USER_AGENT = os.environ.get("OWNER_LENS_SEC_USER_AGENT", "OwnerLens admin@example.com")

raw = SecClient(USER_AGENT).retrieve_company_facts("ADBE").raw_facts
summary = economic_value_summary_from_facts(raw)
summary.overall_economic_value_classification.value

'IMPROVING'

## 1. Annual economics — the latest year

The most recent Feature 2A snapshot answers "did this year's per-share economics improve?"

In [ ]:
snapshots = economic_value_from_facts(raw)
latest = max(snapshots, key=lambda s: s.fiscal_year)
print(f"FY{latest.fiscal_year}: {latest.classification.value}")
print(f"    drivers: {', '.join(d.value for d in latest.drivers)}")

## 2. Multi-year compounding — recent versus long-term

Feature 2B gives a recent (3-year) and a longer (up to 5-year) compounding view. Comparing them separates durable history from recent momentum.

In [2]:
recent, long_term = compounding_views_from_facts(raw)
for label, view in (("recent", recent), ("long-term", long_term)):
    print(f"{label} FY{view.start_fiscal_year}-{view.end_fiscal_year}: {view.classification.value}")
    fcf_ps = view.fcf_per_share_cagr
    print(f"    FCF/share CAGR: {fcf_ps * 100:+.1f}%" if fcf_ps is not None else "    FCF/share CAGR: n/a")

recent FY2022-2025: COMPOUNDING
    FCF/share CAGR: +13.7%
long-term FY2021-2025: COMPOUNDING
    FCF/share CAGR: +12.7%


## 3. Capital allocation — the latest year

Feature 2C answers "did management help per-share owners?"

In [3]:
rows = capital_allocation_from_facts(raw)
latest_row = max(rows, key=lambda r: r.fiscal_year)
print(f"FY{latest_row.fiscal_year}: {latest_row.classification.value}")
print(f"    buyback: {latest_row.buyback_effectiveness.value}")
print(f"    drivers: {', '.join(d.value for d in latest_row.drivers)}")

FY2025: OWNER_FRIENDLY
    buyback: EFFECTIVE_BUYBACKS
    drivers: SHARE_COUNT_SHRANK, EFFECTIVE_BUYBACKS, MEANINGFUL_REPURCHASES, HIGH_SBC_BURDEN, CAPITAL_RETURNED_EXCEEDS_FCF, HIGH_ROIC_CONTEXT, ROIC_IMPROVING, RETAINED_FCF_NEGATIVE, NO_DIVIDEND_PROGRAM


## 4. Areas of agreement

Where the lenses agree, the conclusion is reinforced. For Adobe the per-share engine is compounding (2B), the latest year improved (2A), and capital allocation is owner-friendly (2C) — a coherent, mutually reinforcing positive picture, anchored by a high and rising ROIC.

In [4]:
print("Positive drivers:")
for driver in summary.key_positive_drivers:
    print(f"  + {driver.value}")

Positive drivers:
  + PER_SHARE_CASH_FLOW_COMPOUNDING
  + HIGH_ROIC
  + ROIC_IMPROVING
  + SHARE_COUNT_SHRINKING
  + EFFECTIVE_BUYBACKS
  + OWNER_FRIENDLY_CAPITAL_ALLOCATION


## 5. Areas of tension

Where the lenses disagree, or where a warning is present, the summary surfaces it as a **watch** signal rather than collapsing it into the verdict. High SBC is a real owner cost, and returning more than free cash flow is drawing down the cash cushion — but with shares still shrinking and the balance sheet still net-cash positive, these are watch signals, not guardrails.

In [5]:
print("Watch drivers:")
for driver in summary.key_watch_drivers:
    print(f"  - {driver.value}")
print("\nNegative drivers:")
for driver in summary.key_negative_drivers:
    print(f"  ! {driver.value}")

Watch drivers:
  - HIGH_SBC_BURDEN
  - CAPITAL_RETURNS_EXCEED_FCF
  - DECLINING_NET_CASH_CUSHION

Negative drivers:
  ! MARGINS_CONTRACTING


## 6. The final deterministic OwnerLens summary

One compact, owner-readable conclusion, understandable without any raw financial table.

In [6]:
print(format_economic_value_summary(summary))

ADBE — Economic Value Lens

  Latest annual:       IMPROVING
  Recent compounding:  COMPOUNDING
  Long-term:           COMPOUNDING
  Capital allocation:  OWNER_FRIENDLY

Overall:
  IMPROVING

Core evidence:
  FCF/share CAGR:      +12.7%
  Share-count CAGR:    -2.9%
  ROIC:                61.6%
  Net cash/debt:       +0.4B

Positive:
  + PER_SHARE_CASH_FLOW_COMPOUNDING
  + HIGH_ROIC
  + ROIC_IMPROVING
  + SHARE_COUNT_SHRINKING
  + EFFECTIVE_BUYBACKS
  + OWNER_FRIENDLY_CAPITAL_ALLOCATION

Watch:
  - HIGH_SBC_BURDEN
  - CAPITAL_RETURNS_EXCEED_FCF
  - DECLINING_NET_CASH_CUSHION

Negative:
  ! MARGINS_CONTRACTING


## 7. What this summary does NOT tell us yet

This is an **economic-value** lens, not an investment conclusion. It deliberately stops at the boundary of the next major layer.

It does **not** tell us:

* **Valuation** — whether the stock is cheap or expensive. Strong economics at the wrong price is still a poor investment.
* **Expected return** — no price, no total-shareholder-return, no scenarios.
* **Future competitive durability** — whether the high ROIC and per-share compounding will persist depends on the moat, competition, and technology shifts, none of which are in the financial facts.
* **Capital-allocation intent or skill** — the lens reports observable outcomes, not whether repurchases were made below intrinsic value.

Adobe's *observed* per-share economic value has been improving durably, funded from internal cash while the balance sheet stayed net-cash positive and ROIC stayed high — with a real watch list (SBC burden, distributions above free cash flow, a thinning cash cushion, a slow FCF-margin drift). Whether that translates into a good *investment* is a separate question for a future business-quality and valuation layer.